In [1]:
import numpy as np
import pandas as pd


In [1]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
import pandas as pd
movies=pd.read_csv('movies.csv')
credits=pd.read_csv('credits.csv')

In [7]:
movies = movies.merge(credits, on="title")
print(movies.head())
print(movies.shape)
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
movies.dropna(inplace=True)

      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www.thedarkknightrises.com/   49026   
4          http://movies.disney.com/john-carter   49529   

                                            keywords original_language  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...                en   
2  [{"id": 470, "nam

In [8]:
print(movies.shape)

(4842, 7)


In [9]:
import ast
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
def convert_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

movies['cast'] = movies['cast'].apply(convert_cast)
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [10]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies['tags'] = movies['tags'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))
print(movies['tags'].head())

0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
3    Following the death of District Attorney Harve...
4    John Carter is a war-weary, former military ca...
Name: tags, dtype: object


In [22]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(movies['tags']).toarray()
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)


In [19]:
def recommend(movie):
    index = movies[movies['title'] == movie].index[0]
    distances = list(enumerate(similarity[index]))
    movies_list = sorted(distances, reverse=True, key=lambda x: x[1])[1:6]

    for i in movies_list:
        print(
            movies.iloc[i[0]].title, "->",
            ", ".join(movies.iloc[i[0]].genres)
        )
        
all_genres = []

for genre_list in movies['genres']:
    for g in genre_list:
        all_genres.append(g)

all_genres = sorted(list(set(all_genres)))
def recommend_by_genre(genre):
    # Genre ke andar movies filter karo
    filtered = movies[movies['genres'].apply(lambda x: genre in x)]
    # Top 10 movies
    return list(filtered['title'].head(10))        

In [20]:
recommend('Avatar')

Titan A.E. -> Animation, Action, Science Fiction, Family, Adventure
Small Soldiers -> Comedy, Adventure, Fantasy, Science Fiction, Action
Independence Day -> Action, Adventure, Science Fiction
Aliens vs Predator: Requiem -> Fantasy, Action, Science Fiction, Thriller, Horror
Ender's Game -> Science Fiction, Action, Adventure


In [15]:
recommend_by_genre("Science Fiction")

Avatar
John Carter
Avengers: Age of Ultron
Superman Returns
Man of Steel
The Avengers
Men in Black 3
Captain America: Civil War
Battleship
Jurassic World


In [21]:
import pickle

pickle.dump(movies, open('movies.pkl','wb'))
pickle.dump(similarity, open('similarity.pkl','wb'))